# 1.4 — Equity Analysis (Final Report)

**Author:** Sudipta Sarker

**Purpose:** Generate equity figures for the final report:
- `equity_quintile.png` — Accessibility by income quintile
- `poverty_correlation.png` — Poverty-transit scatter plot
- `counterfactual_rerank.png` — Prescriptive counterfactual reranking

**Data:** Pre-computed parquets in `data/exports/teammate/` OR live from DB via `ctx.equity()`

In [ ]:
# Papermill parameters
city_key = "ywg"
feed_id = "current"
report_name = "final"
save_figures = True
dpi = 200
figure_output_directory = "../../reports/final/figures"

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from ptn_analysis.context import TransitContext
from ptn_analysis.context.reporting import save_placeholder_figure

ctx = TransitContext.from_defaults()
eq = ctx.equity()

figures_dir = Path(figure_output_directory)
figures_dir.mkdir(parents=True, exist_ok=True)

# Pre-computed teammate data (fallback if DB unavailable)
teammate_dir = Path("../data/exports/teammate")
print(f"Context ready: {eq}")
print(f"Figures dir: {figures_dir.resolve()}")

In [ ]:
# === FIGURE 1: equity_quintile.png ===
# Accessibility comparison by income quintile (Q1-Q5)
#
# INSTRUCTIONS FOR SUDIPTA:
# 1. Run eq.travel_time_equity_report() to get quintile data
# 2. Create a bar chart: x=income quintile, y=median_access_score
# 3. Add a second bar for mean_access_score (grouped)
# 4. Annotate the Q1-Q5 gap percentage
# 5. Title: "Transit Accessibility by Income Quintile"
#
# Or load from parquet:
#   report = pd.read_parquet(teammate_dir / "equity_report.parquet")

report = eq.travel_time_equity_report()
if not report.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = range(len(report))
    width = 0.35
    ax.bar([i - width/2 for i in x], report['median_access_score'], width, label='Median Access Score')
    ax.bar([i + width/2 for i in x], report['mean_access_score'], width, label='Mean Access Score')
    ax.set_xlabel('Income Quintile')
    ax.set_ylabel('Transit Access Score')
    ax.set_title('Transit Accessibility by Income Quintile')
    ax.set_xticks(x)
    ax.set_xticklabels(report['income_quintile'])
    ax.legend()
    plt.tight_layout()
    if save_figures:
        fig.savefig(figures_dir / 'equity_quintile.png', dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    print(report[['income_quintile', 'median_access_score', 'mean_access_score', 'neighbourhood_count']])
else:
    save_placeholder_figure('equity_quintile.png',
        'Placeholder: Run eq.travel_time_equity_report()\nBar chart Q1-Q5 median/mean access scores',
        report_name=report_name, figures_dir=str(figures_dir), dpi=dpi, enabled=save_figures)

In [ ]:
# === FIGURE 2: poverty_correlation.png ===
# Poverty-transit scatter with regression line
#
# INSTRUCTIONS FOR SUDIPTA:
# 1. Run eq.poverty_transit_correlation() to get scatter data
# 2. Scatter: x=median_household_income_2020, y=transit_access_score
# 3. Add OLS regression line + R-squared annotation
# 4. Colour by has_poverty_zone if available
# 5. Title: "Income vs Transit Accessibility"
#
# Or load: poverty = pd.read_parquet(teammate_dir / "poverty_correlation.parquet")

poverty = eq.poverty_transit_correlation()
if not poverty.empty and 'transit_access_score' in poverty.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(poverty['median_household_income_2020'], poverty['transit_access_score'],
               alpha=0.6, edgecolors='black', linewidths=0.5)
    ax.set_xlabel('Median Household Income (2020)')
    ax.set_ylabel('Transit Access Score')
    ax.set_title('Income vs Transit Accessibility by Neighbourhood')
    plt.tight_layout()
    if save_figures:
        fig.savefig(figures_dir / 'poverty_correlation.png', dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
else:
    save_placeholder_figure('poverty_correlation.png',
        'Placeholder: Run eq.poverty_transit_correlation()\nScatter + regression line',
        report_name=report_name, figures_dir=str(figures_dir), dpi=dpi, enabled=save_figures)

In [ ]:
# === FIGURE 3: counterfactual_rerank.png ===
# Prescriptive counterfactual: rank change under equity weighting
#
# INSTRUCTIONS FOR SUDIPTA:
# 1. Run eq.equity_weighted_accessibility() to get reranking data
# 2. Horizontal bar chart: y=neighbourhood (top 15 gainers/losers), x=rank_change
# 3. Colour: green for gained priority, red for lost priority
# 4. Title: "Neighbourhood Priority Change Under Equity Weighting"
#
# Or load: cf = pd.read_parquet(teammate_dir / "counterfactual.parquet")

cf = eq.equity_weighted_accessibility()
if not cf.empty:
    top_gainers = cf.nlargest(10, 'rank_change')
    top_losers = cf.nsmallest(10, 'rank_change')
    show = pd.concat([top_gainers, top_losers]).sort_values('rank_change')
    fig, ax = plt.subplots(figsize=(10, 8))
    colours = ['#2ca02c' if v > 0 else '#d62728' for v in show['rank_change']]
    ax.barh(show['neighbourhood'], show['rank_change'], color=colours)
    ax.set_xlabel('Rank Change (positive = gained priority)')
    ax.set_title('Neighbourhood Priority Change Under Equity Weighting')
    ax.axvline(x=0, color='black', linewidth=0.5)
    plt.tight_layout()
    if save_figures:
        fig.savefig(figures_dir / 'counterfactual_rerank.png', dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    reprioritized = (cf['rank_change'].abs() > 0).sum()
    print(f"Reprioritized: {reprioritized}/{len(cf)} ({100*reprioritized/len(cf):.1f}%)")
    print(cf.nlargest(5, 'rank_change')[['neighbourhood', 'rank_change', 'vulnerability_index']])
else:
    save_placeholder_figure('counterfactual_rerank.png',
        'Placeholder: Run eq.equity_weighted_accessibility()\nRank change horizontal bars',
        report_name=report_name, figures_dir=str(figures_dir), dpi=dpi, enabled=save_figures)

In [ ]:
# === Export parquets for teammate use ===
export_dir = Path("../data/exports/teammate")
export_dir.mkdir(parents=True, exist_ok=True)

for name, data in [
    ("equity_report", eq.travel_time_equity_report()),
    ("poverty_correlation", eq.poverty_transit_correlation()),
    ("counterfactual", eq.equity_weighted_accessibility()),
]:
    if not data.empty:
        data.to_parquet(export_dir / f"{name}.parquet")
        data.to_csv(export_dir / f"{name}.csv", index=False)
        print(f"  {name}: {len(data)} rows")
    else:
        print(f"  {name}: EMPTY — check DB")